# Colab 03 - Fine-tune

This notebook runs the reusable stride4 Colab fine-tuning scripts for MMAudio `small_16k`. Large outputs are written to Google Drive under `outputs/`, with logs under `logs/`.

Fresh training:

```bash
!bash scripts/run_colab_finetune_stride4.sh
```

Continue training:

```bash
!bash scripts/run_colab_continue_stride4.sh
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '/content/MMAudio')

from fine_tune.urban_paths import (
    AUDIO_SR,
    CLIP_SEQ_LEN,
    DEFAULT_STRIDE,
    DRIVE_ROOT,
    EVAL_DIR,
    EXT_WEIGHTS_DIR,
    LATENT_SEQ_LEN,
    LOG_DIR,
    MODE,
    MODEL,
    OUTPUT_DIR,
    REPO_ROOT,
    SYNC_SEQ_LEN,
    TEXT_LABEL,
    TEXT_SEQ_LEN,
    TSV_DIR,
    VIDEO_EXTENSIONS,
    WEIGHTS_DIR,
    ensure_drive_dirs,
    latent_dir_for_stride,
    split_video_dirs,
)

STRIDE = DEFAULT_STRIDE
SPLIT_VIDEO_DIRS = split_video_dirs(STRIDE)
LATENTS_ROOT = latent_dir_for_stride(STRIDE)
RUNS_ROOT = OUTPUT_DIR
EVAL_ROOT = EVAL_DIR

ensure_drive_dirs(STRIDE)
for split_dir in SPLIT_VIDEO_DIRS.values():
    split_dir.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_ROOT)
print('Drive root:', DRIVE_ROOT)
print('Stride:', STRIDE)
print('Mode/model:', MODE, MODEL)
print('Latents:', LATENTS_ROOT)
print('Outputs:', RUNS_ROOT)
print('Eval:', EVAL_ROOT)
print('Logs:', LOG_DIR)

In [ ]:
%cd /content/MMAudio

# Keep repo-local paths compatible with scripts that expect ./weights and ./ext_weights,
# while storing the actual large files in Google Drive.
def link_drive_dir(repo_name: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_name
    drive_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if repo_path.is_dir() and not any(repo_path.iterdir()):
            repo_path.rmdir()
        else:
            print(f'{repo_path} exists; linking missing files inside it instead.')
            return
    repo_path.symlink_to(drive_path, target_is_directory=True)
    print(f'Linked {repo_path} -> {drive_path}')


def link_drive_file(repo_rel: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_rel
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if not drive_path.exists():
            repo_path.replace(drive_path)
            print(f'Moved {repo_path} -> {drive_path}')
        else:
            repo_path.unlink()
    repo_path.symlink_to(drive_path)
    print(f'Linked {repo_path} -> {drive_path}')


link_drive_dir('weights', WEIGHTS_DIR)
link_drive_dir('ext_weights', EXT_WEIGHTS_DIR)

for rel, target in {
    'weights/mmaudio_small_16k.pth': WEIGHTS_DIR / 'mmaudio_small_16k.pth',
    'ext_weights/v1-16.pth': EXT_WEIGHTS_DIR / 'v1-16.pth',
    'ext_weights/synchformer_state_dict.pth': EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth',
    'ext_weights/best_netG.pt': EXT_WEIGHTS_DIR / 'best_netG.pt',
    'ext_weights/empty_string.pth': EXT_WEIGHTS_DIR / 'empty_string.pth',
}.items():
    link_drive_file(rel, target)

In [ ]:
%cd /content/MMAudio

# This notebook uses the reusable stride4 scripts.
STRIDE = 4
SPLIT_VIDEO_DIRS = split_video_dirs(STRIDE)
LATENTS_ROOT = latent_dir_for_stride(STRIDE)
RUNS_ROOT = OUTPUT_DIR
EVAL_ROOT = EVAL_DIR

ensure_drive_dirs(STRIDE)
for split_dir in SPLIT_VIDEO_DIRS.values():
    split_dir.mkdir(parents=True, exist_ok=True)

EXP_ID = 'urban_ft_8s_stride4_18k'
RUN_DIR = RUNS_ROOT / EXP_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Stride:', STRIDE)
print('Latents:', LATENTS_ROOT)
print('Run dir:', RUN_DIR)


In [ ]:
%cd /content/MMAudio

import pandas as pd
from tensordict import TensorDict

required_files = [
    WEIGHTS_DIR / 'mmaudio_small_16k.pth',
    EXT_WEIGHTS_DIR / 'v1-16.pth',
    EXT_WEIGHTS_DIR / 'best_netG.pt',
    EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth',
    EXT_WEIGHTS_DIR / 'empty_string.pth',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

script_tsv_dir = TSV_DIR / f'stride{STRIDE}'
active_tsv_dir = script_tsv_dir if (script_tsv_dir / 'video_train_ft.tsv').exists() else TSV_DIR
print('TSV dir:', active_tsv_dir)

for split in ['train', 'val', 'test']:
    tsv_path = active_tsv_dir / f'video_{split}_ft.tsv'
    memmap_dir = LATENTS_ROOT / split / 'training_latents'
    if not tsv_path.exists():
        raise FileNotFoundError(tsv_path)
    if not memmap_dir.exists():
        raise FileNotFoundError(memmap_dir)

    td = TensorDict.load_memmap(memmap_dir)
    n_rows = len(pd.read_csv(tsv_path, sep='\t'))
    print(f'{split}: {n_rows} rows, mean shape={tuple(td["mean"].shape)}')
    assert td['mean'].shape[0] == n_rows
    assert td['mean'].shape[1] == LATENT_SEQ_LEN


In [ ]:
%cd /content/MMAudio

!bash scripts/run_colab_finetune_stride4.sh


In [ ]:
%cd /content/MMAudio

# To continue from the default latest checkpoint, run:
# !bash scripts/run_colab_continue_stride4.sh

# To continue from a specific checkpoint, set CHECKPOINT_PATH before bash:
# !CHECKPOINT_PATH="/content/drive/MyDrive/MMAudio_Yuqi/outputs/<exp_id>/<checkpoint>.pth" bash scripts/run_colab_continue_stride4.sh


In [ ]:
from pathlib import Path

print('Run artifacts:')
for p in sorted(RUN_DIR.glob('*')):
    print(p)

print('\nUseful outputs:')
print('Last weights:', RUN_DIR / f'{EXP_ID}_last.pth')
print('Last checkpoint:', RUN_DIR / f'{EXP_ID}_ckpt_last.pth')
print('EMA final, if enabled:', RUN_DIR / f'{EXP_ID}_ema_final.pth')